In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
import json
import random
import csv
from pathlib import Path
import sys
from dotenv import load_dotenv

project_root = Path.cwd()
while not (project_root / "utils").is_dir() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
load_dotenv(project_root / ".env", override=True)

from utils.seku_crd_corpus_utils import *
# from prediction.predict_ner import *
import random

### SEKU Dataset ###

In [15]:
NB_ABSTRACTS = 6000
# collect abstracts in CSV for synthetic data generation for training
ABSTRACT_CSV = f"/Work/Experiments/seku-crd-citation-processor/data/training_data/synthetic_dgft_abstracts_{NB_ABSTRACTS}.csv"
CITATION_FRACTION = 0.3 # for each paper collect CITATION_FRACTION number of its reference papers, for each of these samples reference paper collect abstract

In [16]:
def write_abstracts_csv(records, out_path):
    """
    Write sampled ABSTRACT_ARCHIVE records to a CSV with columns:
    paper_id, data_source, abstract.
    """
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["paper_id","abstract"])
        for rec in records:
            writer.writerow([rec["paper_id"],rec["abstract"]])

In [5]:
paper_index = load_paper_index()
all_main_paper_ids = list(paper_index.keys())
nb_main_papers = len(all_main_paper_ids)
print(f'Total {nb_main_papers} Main papers')

Loading paper index: 166310line [00:32, 5087.13line/s] 


Loaded 166310 papers from /Work/Experiments/citation-processor-ranker/data/unarxive/processed/paper_index_2024.jsonl (6496.9 MB in memory)
Total 166310 Main papers


In [18]:
count = 0
all_records = []
while count <= NB_ABSTRACTS:
    # pick one random main paper
    rand_idx = random.randint(0,nb_main_papers-1)
    rand_paper_id = all_main_paper_ids[rand_idx]
    # print(f"Random Main paper Index:{rand_idx} [Corpus ID: {rand_paper_id}]")
    main_data = paper_index[rand_paper_id] # data from paper index file
    # collect abstract for main paper
    main_paper_data = get_paper_from_s3(rand_paper_id)
    main_title,main_abstract = get_title_and_abstract(main_paper_data)
    all_records.append({"paper_id":rand_paper_id,"abstract":main_abstract})
    count += 1
    # now collect reference papers
    in_corpus_refs_ids = main_data["in_corpus_references"]
    sample_nb = int(len(in_corpus_refs_ids)*CITATION_FRACTION)
    sample_ref_ids = random.sample(in_corpus_refs_ids,sample_nb)
    results = await get_papers_from_s3(sample_ref_ids)
    for ref_paper_id, data in results.items():
        if data is not None:
            ref_title,ref_abstract = get_title_and_abstract(data)
            all_records.append({"paper_id":ref_paper_id,"abstract":ref_abstract})
            count += 1
    print(f"Total {count}/{NB_ABSTRACTS} collected")
write_abstracts_csv(records=all_records,out_path=ABSTRACT_CSV)

Fetching papers: 100%|██████████| 4/4 [00:00<00:00,  7.18it/s]


Total 5/6000 collected


Fetching papers: 100%|██████████| 8/8 [00:00<00:00, 14.05it/s]


Total 14/6000 collected


Fetching papers: 100%|██████████| 12/12 [00:00<00:00, 21.86it/s]


Total 27/6000 collected


... (1,180 outputs left out)

Fetching papers: 100%|██████████| 6/6 [00:00<00:00,  6.53it/s]


Total 5991/6000 collected


Fetching papers: 100%|██████████| 12/12 [00:00<00:00, 14.77it/s]


Total 6004/6000 collected


In [19]:
len(all_records)

6004